# RAG-Vorlesungs-Tutor — Client

Dünner Client für das Backend. **Agent, Retrieval und LLM laufen im Server**
(`server/api.py`) und werden beim API-Start aufgebaut. Dieses Notebook lädt nur
PDFs hoch und stellt Fragen – alles über HTTP.

**Server vorher starten** (eigenes Terminal):
```bash
cd server
python api.py
```

## Setup

In [36]:
import requests
from pathlib import Path

API_URL = "http://127.0.0.1:8000"

## PDFs hochladen (`/ingest`)

Legt die PDFs aus `input_pdfs/` per Upload beim Backend ab; dort werden sie mit
docling in Markdown gewandelt, gechunkt und in die Vektor-DB geschrieben.
Der erste Upload lädt einmalig die docling-Modelle → das kann dauern.

In [37]:
PDF_DIR = Path.cwd() / "input_pdfs"
pdf_paths = sorted(PDF_DIR.glob("*.pdf"))
print(f"{len(pdf_paths)} PDF(s):", [p.name for p in pdf_paths])

if not pdf_paths:
    print("Keine PDFs gefunden – lege welche in", PDF_DIR)
else:
    handles = [open(p, "rb") for p in pdf_paths]
    try:
        files = [("files", (p.name, fh, "application/pdf"))
                 for p, fh in zip(pdf_paths, handles)]
        # formulas=true aktiviert die Formel-/LaTeX-Interpretation (langsamer).
        # Optional zusaetzlich: "ocr": "true" (gescannte PDFs), "delete_pdfs": "false".
        resp = requests.post(f"{API_URL}/ingest", files=files, data={"formulas": "true"})
        print("Status:", resp.status_code)
        print(resp.json())
    finally:
        for fh in handles:
            fh.close()

0 PDF(s): []
Keine PDFs gefunden – lege welche in /home/tim/Dokumente/NLP/rag-lecture-tutor/input_pdfs


## Frage stellen (`/ask`)

Schickt die Frage an den Backend-Agenten und zeigt dessen Antwort.

In [38]:
# Das ausführen dieser Zelle setzt die Konversations
messages = []

In [39]:
# Um eine Followup Frage zu stellen nur diese Zelle ausführen.
frage = input("Frage: ")
messages.append({
        "role": "user",
        "content": frage
    })
print(f"{frage}")
resp = requests.post(f"{API_URL}/ask", json={"messages": messages})
if resp.status_code == 200:
    answer = resp.json()["answer"]
    print("Agent:", answer)

    messages.append({
        "role": "assistant",
        "content": answer
    })
else:
    print("Fehler:", resp.status_code, resp.text)

Was ist eine Linearkombination und Erzeugnis?
Agent: **Linearkombination**

Eine Linearkombination ist eine endliche Summe von Vektoren aus einem Vektorraum, bei der jeder Vektor mit einem Skalar aus dem zugrunde liegenden Körper multipliziert wird.  
Formell: Für einen \(K\)-Vektorraum \(V\) und Vektoren \(v_1,\dots ,v_n\in V\) sowie Skalare \(\lambda_1,\dots ,\lambda_n\in K\) gilt

\[
\lambda_1\cdot v_1 + \dots + \lambda_n\cdot v_n
   = \sum_{i=1}^{n}\lambda_i\cdot v_i .
\]

Die Summe \(\sum_{i=1}^{n}\lambda_i v_i\) ist genau das, was wir als **Linearkombination** der Vektoren \(v_1,\dots ,v_n\) bezeichnen.

---

**Erzeugnis (Spann)**

Das Erzeugnis, auch Spann genannt, ist die Menge aller Linearkombinationen einer gegebenen Menge von Vektoren.  
Wenn \(M\subseteq V\) ein (endliches) Vektorsystem ist, dann ist

\[
\operatorname{span}(M)=\{\,\lambda_1 v_1+\dots+\lambda_k v_k
        \mid v_1,\dots ,v_k\in M,\;\lambda_1,\dots ,\lambda_k\in K\,\}
\]

die kleinste Untergruppe von \(V\), 